# 🧪 Notebook 5 — Test du Modèle
**ISIC Project | Segmentation de Lésions Cutanées**

| Mode | Description |
|------|-------------|
| ✅ **Mode 1** | Tester sur **une seule image** |
| ✅ **Mode 2** | Tester sur **un dossier** (batch) |
| ✅ **Mode 3** | Évaluation complète sur le **jeu de test** |

**Métriques calculées** : Accuracy, Dice, IoU, Précision, Recall, F1, Spécificité

## ⚙️ Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path
PROJECT_PATH  = '/content/drive/MyDrive/ISIC_Project'
IMAGES_PATH   = os.path.join(PROJECT_PATH, 'data', 'Images')
MASQUES_PATH  = os.path.join(PROJECT_PATH, 'data', 'Masques')
OUTPUTS_PATH  = os.path.join(PROJECT_PATH, 'outputs')
MODEL_PATH    = os.path.join(OUTPUTS_PATH, 'best_unet_isic.pth')
PREDS_PATH    = os.path.join(OUTPUTS_PATH, 'predictions')
os.makedirs(PREDS_PATH, exist_ok=True)

if os.path.exists(MODEL_PATH):
    print(f'✅ Modèle trouvé ({os.path.getsize(MODEL_PATH)/1024/1024:.1f} MB)')
else:
    print('❌ Lance d\'abord le notebook 03 !')

In [ ]:
!pip install -q albumentations opencv-python-headless scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm
import json, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Imports OK | Device : {device}')

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        self.pool = nn.MaxPool2d(2,2)
        for f in features:
            self.downs.append(DoubleConv(in_channels, f)); in_channels = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_channels, 1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); skip = skips[i//2]
            if x.shape != skip.shape: x = F.interpolate(x, size=skip.shape[2:])
            x = self.ups[i+1](torch.cat([skip, x], dim=1))
        return self.final(x)

model = UNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

IMG_SIZE = 256
THRESHOLD = 0.5
infer_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
print('✅ Modèle prêt !')

In [ ]:
def predict_single(image_path, threshold=THRESHOLD):
    img_orig = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w     = img_orig.shape[:2]
    tensor   = infer_transform(image=img_orig)['image'].unsqueeze(0).float().to(device)
    with torch.no_grad():
        logit    = model(tensor)
        prob_map = torch.sigmoid(logit).squeeze().cpu().numpy()
        mask_256 = (prob_map > threshold).astype(np.uint8)
    mask_orig = cv2.resize(mask_256, (w, h), interpolation=cv2.INTER_NEAREST)
    return img_orig, prob_map, mask_256, mask_orig


def compute_all_metrics(pred, true, smooth=1e-6):
    """
    Calcule toutes les métriques à partir de deux masques numpy binaires.
    Inclut l'Accuracy avec une note sur le déséquilibre de classes.
    """
    p = pred.flatten().astype(float)
    t = true.flatten().astype(float)
    tp = (p * t).sum()
    tn = ((1-p) * (1-t)).sum()
    fp = (p * (1-t)).sum()
    fn = ((1-p) * t).sum()
    total = tp + tn + fp + fn

    accuracy    = (tp+tn) / (total+smooth)
    dice        = (2*tp+smooth) / (2*tp+fp+fn+smooth)
    iou         = (tp+smooth) / (tp+fp+fn+smooth)
    precision   = (tp+smooth) / (tp+fp+smooth)
    recall      = (tp+smooth) / (tp+fn+smooth)
    f1          = dice
    specificity = (tn+smooth) / (tn+fp+smooth)

    return {
        'Accuracy'    : accuracy,
        'Dice'        : dice,
        'IoU'         : iou,
        'Précision'   : precision,
        'Recall'      : recall,
        'F1'          : f1,
        'Spécificité' : specificity,
    }


def show_prediction(img_orig, prob_map, mask_pred, mask_true=None, title='', metrics=None):
    has_gt = mask_true is not None
    n_cols = 5 if has_gt else 3
    fig, axes = plt.subplots(1, n_cols, figsize=(4*n_cols, 4.5))

    img_r   = cv2.resize(img_orig, (256, 256))
    overlay = img_r.copy()
    overlay[mask_pred > 0] = [255, 80, 80]
    overlay = cv2.addWeighted(img_r, 0.6, overlay, 0.4, 0)

    axes[0].imshow(img_r);            axes[0].set_title('Image');         axes[0].axis('off')
    axes[1].imshow(prob_map, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title('Probabilité'); axes[1].axis('off')
    axes[2].imshow(overlay);          axes[2].set_title('Overlay');       axes[2].axis('off')

    if has_gt:
        mtr = cv2.resize(mask_true.astype(np.uint8), (256,256), interpolation=cv2.INTER_NEAREST)
        diff = np.zeros((*mask_pred.shape, 3), dtype=np.uint8)
        diff[(mask_pred==1)&(mtr==1)] = [0,200,0]
        diff[(mask_pred==1)&(mtr==0)] = [255,80,80]
        diff[(mask_pred==0)&(mtr==1)] = [80,80,255]
        axes[3].imshow(mtr, cmap='gray'); axes[3].set_title('Masque réel'); axes[3].axis('off')
        axes[4].imshow(diff)
        legend = [
            mpatches.Patch(color='green', label='TP'),
            mpatches.Patch(color='red',   label='FP'),
            mpatches.Patch(color='blue',  label='FN'),
        ]
        axes[4].legend(handles=legend, loc='lower right', fontsize=7)
        axes[4].set_title('Erreurs'); axes[4].axis('off')

    if metrics:
        acc_str  = f"Acc={metrics['Accuracy']:.3f}  "
        dice_str = f"Dice={metrics['Dice']:.3f}  IoU={metrics['IoU']:.3f}\n"
        pr_str   = f"Prec={metrics['Précision']:.3f}  Rec={metrics['Recall']:.3f}"
        fig.suptitle(f'{title}\n{acc_str}{dice_str}{pr_str}', fontsize=10, fontweight='bold')
    else:
        fig.suptitle(title, fontsize=11)
    plt.tight_layout(); plt.show()


print('✅ Fonctions prêtes (avec Accuracy) !')

---
## 🔵 MODE 1 — Une seule image

In [ ]:
# ✏️ Modifie ces chemins
IMAGE_PATH = '/content/drive/MyDrive/ISIC_Project/data/Images/ISIC_0024306.jpg'
MASK_PATH  = '/content/drive/MyDrive/ISIC_Project/data/Masques/ISIC_0024306_segmentation.png'  # ou None

if not os.path.exists(IMAGE_PATH):
    print('❌ Image introuvable — vérifie IMAGE_PATH')
else:
    img_orig, prob_map, mask_pred, mask_orig = predict_single(IMAGE_PATH)

    mask_true, metrics = None, None
    if MASK_PATH and os.path.exists(MASK_PATH):
        mask_true_full = (cv2.imread(MASK_PATH, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        mask_true_256  = cv2.resize(mask_true_full, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        metrics = compute_all_metrics(mask_pred, mask_true_256)

        print('📊 Métriques complètes :')
        for k, v in metrics.items():
            note = '  ← MÉTRIQUE PRINCIPALE' if k=='Dice' else (
                   '  ← ATTENTION : gonflé par déséquilibre' if k=='Accuracy' else '')
            print(f'   {k:<14} : {v:.4f}{note}')

    print(f'\n🔍 Surface lésion prédite : {mask_pred.sum()/mask_pred.size*100:.1f}%')
    show_prediction(img_orig, prob_map, mask_pred, mask_true_full if mask_true_full is not None else None,
                    'Mode 1 — Image unique', metrics)

---
## 🟡 MODE 1B — Upload depuis ton PC

In [ ]:
from google.colab import files
print('📤 Uploade une image...')
uploaded = files.upload()

for filename, data in uploaded.items():
    tmp = f'/tmp/{filename}'
    with open(tmp, 'wb') as f: f.write(data)
    img_orig, prob_map, mask_pred, _ = predict_single(tmp)
    print(f'\n🔍 {filename} — Surface lésion : {mask_pred.sum()/mask_pred.size*100:.1f}%')
    show_prediction(img_orig, prob_map, mask_pred, title=f'Upload — {filename}')

---
## 🔴 MODE 3 — Évaluation complète sur le jeu de test

In [ ]:
from sklearn.model_selection import train_test_split

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
all_images = sorted([p for p in Path(IMAGES_PATH).glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS])
all_masks  = sorted([p for p in Path(MASQUES_PATH).glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS])

_, temp_imgs, _, temp_msks = train_test_split(all_images, all_masks, test_size=0.3, random_state=42)
_, test_imgs, _, test_msks = train_test_split(temp_imgs, temp_msks, test_size=0.5, random_state=42)
print(f'✅ Test set : {len(test_imgs)} images')

# ── Évaluation ───────────────────────────────────────────────────────────────
all_results = []

for img_path, msk_path in tqdm(zip(test_imgs, test_msks), total=len(test_imgs), desc='Test'):
    img_orig, prob_map, mask_pred, _ = predict_single(img_path)
    mask_true = cv2.resize(
        (cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8),
        (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    m = compute_all_metrics(mask_pred, mask_true)
    m['filename']   = img_path.name
    m['lesion_pct'] = float(mask_pred.sum() / mask_pred.size * 100)
    all_results.append(m)

# ── Résumé ───────────────────────────────────────────────────────────────────
metric_keys = ['Accuracy','Dice','IoU','Précision','Recall','F1','Spécificité']
means = {k: np.mean([r[k] for r in all_results]) for k in metric_keys}
stds  = {k: np.std( [r[k] for r in all_results]) for k in metric_keys}

print('\n' + '='*65)
print(f'  {"Métrique":<14} {"Moyenne":<10} {"Écart-type":<12} Interprétation')
print('='*65)
notes = {
    'Accuracy'   : '⚠️  Gonflé par déséquilibre classes',
    'Dice'       : '⭐ MÉTRIQUE PRINCIPALE',
    'IoU'        : '   Plus stricte que Dice',
    'Précision'  : '   Qualité prédictions +',
    'Recall'     : '   Taux détection lésions',
    'F1'         : '   = Dice Score',
    'Spécificité': '   Vrais négatifs corrects',
}
for k in metric_keys:
    print(f'  {k:<14} {means[k]:.4f}    ±{stds[k]:.4f}      {notes[k]}')
print('='*65)

# Sauvegarde
with open(os.path.join(OUTPUTS_PATH, 'test_metrics_complet.json'), 'w') as f:
    json.dump({'means': means, 'stds': stds, 'per_image': all_results}, f, indent=2)
print('\n💾 Sauvegardé : outputs/test_metrics_complet.json')

In [ ]:
# ── Graphique comparatif Accuracy vs Dice ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Toutes les métriques
colors = ['#F26419','#0E7C7B','#14B8A6','#334155','#3B82F6','#8B5CF6','#64748B']
bars = axes[0].bar(metric_keys, [means[k] for k in metric_keys],
                   color=colors, edgecolor='white',
                   yerr=[stds[k] for k in metric_keys], capsize=4)
for bar, k in zip(bars, metric_keys):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015,
                 f'{means[k]:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[0].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Seuil 0.8')
axes[0].set_ylim(0, 1.2); axes[0].set_title('Toutes les métriques', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# 2. Accuracy vs Dice — pourquoi Accuracy est trompeuse
cats = ['U-Net\n(notre modèle)', 'Modèle naïf\n(tout = 0)', 'Modèle parfait']
acc_v  = [means['Accuracy'], 0.95, 1.0]
dice_v = [means['Dice'],     0.0,  1.0]
x = np.arange(3); w = 0.35
b1 = axes[1].bar(x-w/2, acc_v,  w, label='Accuracy',   color='#F26419', alpha=0.85, edgecolor='white')
b2 = axes[1].bar(x+w/2, dice_v, w, label='Dice Score', color='#0E7C7B', alpha=0.85, edgecolor='white')
for bar, v in zip(b1, acc_v):  axes[1].text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', fontsize=9, fontweight='bold', color='#F26419')
for bar, v in zip(b2, dice_v): axes[1].text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', fontsize=9, fontweight='bold', color='#0E7C7B')
axes[1].set_ylim(0, 1.2); axes[1].set_xticks(x); axes[1].set_xticklabels(cats)
axes[1].set_title('Accuracy vs Dice — Attention !', fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
axes[1].annotate('95% Accuracy\nmais Dice=0.0 !\n(rien détecté)', xy=(1, 0.05),
                 xytext=(1.4, 0.4), fontsize=8, color='red', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='red'))

# 3. Dice Score vs Surface lésion
lesion_pcts = [r['lesion_pct'] for r in all_results]
dice_scores = [r['Dice']       for r in all_results]
axes[2].scatter(lesion_pcts, dice_scores, alpha=0.5, color='#0E7C7B', s=20)
axes[2].set_xlabel('Surface lésion (%)'); axes[2].set_ylabel('Dice Score')
axes[2].set_title('Dice vs Surface lésion', fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Analyse complète — Jeu de Test ISIC', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'analyse_complete.png'), dpi=150)
plt.show()
print('💾 Sauvegardé : outputs/analyse_complete.png')

## 🎛️ BONUS — Meilleur seuil pour chaque métrique

In [ ]:
thresholds  = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
sample_imgs = test_imgs[:30]
sample_msks = test_msks[:30]

thr_results = {thr: {k: [] for k in metric_keys} for thr in thresholds}

for thr in thresholds:
    for img_path, msk_path in zip(sample_imgs, sample_msks):
        _, _, mask_pred, _ = predict_single(img_path, threshold=thr)
        mask_true = cv2.resize(
            (cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8),
            (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        m = compute_all_metrics(mask_pred, mask_true)
        for k in metric_keys: thr_results[thr][k].append(m[k])

print(f'\n{"Seuil":<8}', end='')
for k in metric_keys: print(f'{k:<12}', end='')
print()
print('-' * (8 + 12*len(metric_keys)))

for thr in thresholds:
    print(f'{thr:<8.2f}', end='')
    for k in metric_keys:
        val = np.mean(thr_results[thr][k])
        print(f'{val:<12.4f}', end='')
    print()

# Meilleur seuil selon Dice
best_thr = max(thresholds, key=lambda t: np.mean(thr_results[t]['Dice']))
print(f'\n🏆 Meilleur seuil selon Dice : {best_thr}')
print(f'   Dice moyen : {np.mean(thr_results[best_thr]["Dice"]):.4f}')

# Graphique
fig, ax = plt.subplots(figsize=(10, 5))
colors_plot = {'Accuracy':'#F26419','Dice':'#0E7C7B','IoU':'#14B8A6',
               'Précision':'#334155','Recall':'#3B82F6','F1':'#8B5CF6','Spécificité':'#64748B'}
for k in metric_keys:
    vals = [np.mean(thr_results[t][k]) for t in thresholds]
    style = '-' if k == 'Dice' else ('--' if k == 'Accuracy' else ':')
    lw    = 3 if k in ('Dice','Accuracy') else 1.5
    ax.plot(thresholds, vals, style, color=colors_plot[k], label=k, linewidth=lw)
ax.axvline(best_thr, color='red', lw=2, linestyle='--', label=f'Meilleur seuil ({best_thr})')
ax.set_xlabel('Seuil'); ax.set_ylabel('Score')
ax.set_title('Évolution des métriques selon le seuil de décision', fontweight='bold')
ax.legend(loc='lower left', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'threshold_all_metrics.png'), dpi=150)
plt.show()
print('💾 Sauvegardé : outputs/threshold_all_metrics.png')